# Gaussian indirect results across trained networks

The saved n=10 runs are compared across six network settings for S=40 (2D) and S=80 (4D). All figures use the existing Gaussian plotting function. No inference, thresholds, diagnostics, or result CSVs are generated here.


## Imports


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (
        parent
        for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (parent / "examples" / "gaussian").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or a directory within it.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.gaussian.analysis.plot import load_runs, plot_metric_grid


## Result directories


In [ ]:
GAUSSIAN_RESULT_DIR = ROOT / "examples" / "gaussian" / "results"
RESULT_DIRS = {
    "default": GAUSSIAN_RESULT_DIR,
    "seed_2026": GAUSSIAN_RESULT_DIR / "ablation" / "seed_2026",
    "fixed_lr": GAUSSIAN_RESULT_DIR / "ablation" / "fixed_lr",
    "no_standardize": GAUSSIAN_RESULT_DIR / "ablation" / "no_standardize",
    "summary_only": GAUSSIAN_RESULT_DIR / "ablation" / "summary_only",
    "inference_only": GAUSSIAN_RESULT_DIR / "ablation" / "inference_only",
}
SETTING_LABELS = {
    "default": "Default",
    "seed_2026": "Different seed",
    "fixed_lr": "Fixed learning rate",
    "no_standardize": "No standardization",
    "summary_only": "Summary-only standardization",
    "inference_only": "Inference-only standardization",
}
SUMMARY_DIMS = (40, 80)
RUN_LABELS = {40: "S=2D", 80: "S=4D"}
KEYS = ["source_model", "id", "candidate_model"]

# Edit yscale: "linear" or "symlog".
# linthresh sets the linear interval [-linthresh, linthresh]; logml uses log10 units.
# With linthresh a power of 10, linscale=.9 gives equal spacing through zero and decade ticks.
# ylim=None fits all rows in the current figure; fixed limits also match across settings.
# yticks=None uses clean automatic ticks; pass a list to set ticks manually.
Y_AXIS = {
    "posterior_mmd": dict(yscale="linear", linthresh=.1, linscale=.9,
                          ylim=(0., 1.5), yticks=None),
    "logml": dict(yscale="symlog", linthresh=1., linscale=.9,
                  ylim=(-1000., 10.), yticks=None),
    "pmp": dict(yscale="linear", linthresh=.1, linscale=.9,
                ylim=(-1.05, 1.05), yticks=None),
}


## Load saved indirect runs


In [3]:
all_runs = {
    setting: load_runs(
        method="indirect",
        num_obs=10,
        summary_dims=list(SUMMARY_DIMS),
        result_dir=result_dir,
    )
    for setting, result_dir in RESULT_DIRS.items()
}
for setting, runs in all_runs.items():
    if set(runs) != set(RUN_LABELS.values()):
        raise ValueError(f"{setting}: expected only S=2D and S=4D, got {list(runs)}")
    print(f"{SETTING_LABELS[setting]}: {', '.join(runs)}")


Default: S=2D, S=4D
Different seed: S=2D, S=4D
Fixed learning rate: S=2D, S=4D
No standardization: S=2D, S=4D
Summary-only standardization: S=2D, S=4D
Inference-only standardization: S=2D, S=4D


## Check paired datasets and shared gold standards


In [4]:
for dim, run_label in RUN_LABELS.items():
    baseline = all_runs["default"][run_label].set_index(KEYS).sort_index()
    for setting in RESULT_DIRS:
        aligned = all_runs[setting][run_label].set_index(KEYS).sort_index()
        if not aligned.index.equals(baseline.index):
            raise ValueError(f"{setting}, S={dim}: source_model/id/candidate_model rows differ from default")
        for column in ("gold_logml", "gold_pmp"):
            if column not in baseline or column not in aligned:
                raise ValueError(f"{setting}, S={dim}: missing {column}")
            if not np.isfinite(baseline[column].to_numpy(float)).all():
                raise ValueError(f"default, S={dim}: {column} contains non-finite values")
            try:
                pd.testing.assert_series_equal(baseline[column], aligned[column], check_exact=True)
            except AssertionError as exc:
                raise ValueError(f"{setting}, S={dim}: {column} differs from default") from exc

print(f"All {len(RESULT_DIRS)} settings have identical dataset keys and gold_logml/gold_pmp values at S=40 and S=80.")


All 6 settings have identical dataset keys and gold_logml/gold_pmp values at S=40 and S=80.


## Posterior MMD figures


### Default


In [ ]:
fig, _ = plot_metric_grid(all_runs["default"], metric="posterior_mmd", **Y_AXIS["posterior_mmd"])
display(fig)
plt.close(fig)


### Different seed


In [ ]:
fig, _ = plot_metric_grid(all_runs["seed_2026"], metric="posterior_mmd", **Y_AXIS["posterior_mmd"])
display(fig)
plt.close(fig)


### Fixed learning rate


In [ ]:
fig, _ = plot_metric_grid(all_runs["fixed_lr"], metric="posterior_mmd", **Y_AXIS["posterior_mmd"])
display(fig)
plt.close(fig)


### No standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["no_standardize"], metric="posterior_mmd", **Y_AXIS["posterior_mmd"])
display(fig)
plt.close(fig)


### Summary-only standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["summary_only"], metric="posterior_mmd", **Y_AXIS["posterior_mmd"])
display(fig)
plt.close(fig)


### Inference-only standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["inference_only"], metric="posterior_mmd", **Y_AXIS["posterior_mmd"])
display(fig)
plt.close(fig)


## Log10 marginal-likelihood error figures


### Default


In [ ]:
fig, _ = plot_metric_grid(all_runs["default"], metric="logml", **Y_AXIS["logml"])
display(fig)
plt.close(fig)


### Different seed


In [ ]:
fig, _ = plot_metric_grid(all_runs["seed_2026"], metric="logml", **Y_AXIS["logml"])
display(fig)
plt.close(fig)


### Fixed learning rate


In [ ]:
fig, _ = plot_metric_grid(all_runs["fixed_lr"], metric="logml", **Y_AXIS["logml"])
display(fig)
plt.close(fig)


### No standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["no_standardize"], metric="logml", **Y_AXIS["logml"])
display(fig)
plt.close(fig)


### Summary-only standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["summary_only"], metric="logml", **Y_AXIS["logml"])
display(fig)
plt.close(fig)


### Inference-only standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["inference_only"], metric="logml", **Y_AXIS["logml"])
display(fig)
plt.close(fig)


## PMP error figures


The common PMP y-axis range is −1.05 to 1.05. Marker and high-surprise definitions come from the existing plotting code.


### Default


In [ ]:
fig, _ = plot_metric_grid(all_runs["default"], metric="pmp", **Y_AXIS["pmp"])
display(fig)
plt.close(fig)


### Different seed


In [ ]:
fig, _ = plot_metric_grid(all_runs["seed_2026"], metric="pmp", **Y_AXIS["pmp"])
display(fig)
plt.close(fig)


### Fixed learning rate


In [ ]:
fig, _ = plot_metric_grid(all_runs["fixed_lr"], metric="pmp", **Y_AXIS["pmp"])
display(fig)
plt.close(fig)


### No standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["no_standardize"], metric="pmp", **Y_AXIS["pmp"])
display(fig)
plt.close(fig)


### Summary-only standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["summary_only"], metric="pmp", **Y_AXIS["pmp"])
display(fig)
plt.close(fig)


### Inference-only standardization


In [ ]:
fig, _ = plot_metric_grid(all_runs["inference_only"], metric="pmp", **Y_AXIS["pmp"])
display(fig)
plt.close(fig)


## Inference accuracy summary

Medians use the saved result columns. Posterior MMD, absolute logML error, and absolute PMP error summarize candidate-model rows. PMP L1 error is counted once per observed dataset (source_model, id).


In [8]:
summary_rows = []
for setting, runs in all_runs.items():
    for dim, run_label in RUN_LABELS.items():
        frame = runs[run_label]
        dataset_keys = ["source_model", "id"]
        if not frame.groupby(dataset_keys)["pmp_l1_error"].nunique(dropna=False).eq(1).all():
            raise ValueError(f"{setting}, S={dim}: pmp_l1_error differs across candidate rows")
        dataset_pmp_l1 = frame.drop_duplicates(dataset_keys)["pmp_l1_error"]
        summary_rows.append({
            "setting": SETTING_LABELS[setting],
            "summary_dim": dim,
            "median posterior_mmd": frame["posterior_mmd"].median(),
            "median absolute_logml_error": frame["absolute_logml_error"].median(),
            "median absolute_pmp_error": frame["absolute_pmp_error"].median(),
            "median dataset pmp_l1_error": dataset_pmp_l1.median(),
        })

inference_summary = pd.DataFrame(summary_rows).set_index(["setting", "summary_dim"])
display(inference_summary.round(4))


median posterior_mmd  ...  median dataset pmp_l1_error
setting                        summary_dim                        ...                             
Default                        40                         0.4853  ...                       0.0048
                               80                         0.4864  ...                       0.0046
Different seed                 40                         0.4480  ...                       0.0049
                               80                         0.4719  ...                       0.0045
Fixed learning rate            40                         0.4001  ...                       0.0055
                               80                         0.3720  ...                       0.0053
No standardization             40                         0.3429  ...                       0.0029
                               80                         0.3494  ...                       0.0032
Summary-only standardization   40                         0.5216  ...                       0.0048
                               80                         0.5309  ...                       0.0047
Inference-only standardization 40                         0.3498  ...                       0.0029
                               80                         0.3508  ...                       0.0032

[12 rows x 4 columns]

## Paired differences from Default

Each variant row is matched to Default by source_model, id, and candidate_model. The table shows the median signed change and the 90th percentile of its absolute magnitude for each saved metric. Signed logML errors retain their saved units.


In [9]:
paired_metrics = ("posterior_mmd", "signed_logml_error", "signed_pmp_error")
paired_rows = []
for setting in RESULT_DIRS:
    if setting == "default":
        continue
    for dim, run_label in RUN_LABELS.items():
        baseline = all_runs["default"][run_label].set_index(KEYS).sort_index()
        variant = all_runs[setting][run_label].set_index(KEYS).sort_index()
        delta = variant[list(paired_metrics)] - baseline[list(paired_metrics)]
        row = {"variant": SETTING_LABELS[setting], "summary_dim": dim}
        for metric in paired_metrics:
            row[f"median Δ {metric}"] = delta[metric].median()
            row[f"p90 |Δ {metric}|"] = delta[metric].abs().quantile(0.9)
        paired_rows.append(row)

paired_summary = pd.DataFrame(paired_rows).set_index(["variant", "summary_dim"])
display(paired_summary.round(4))


median Δ posterior_mmd  ...  p90 |Δ signed_pmp_error|
variant                        summary_dim                          ...                          
Different seed                 40                          -0.0055  ...                    0.0737
                               80                          -0.0062  ...                    0.0657
Fixed learning rate            40                          -0.0162  ...                    0.0335
                               80                          -0.0399  ...                    0.0402
No standardization             40                          -0.0273  ...                    0.0420
                               80                          -0.0332  ...                    0.0326
Summary-only standardization   40                           0.0009  ...                    0.0072
                               80                           0.0012  ...                    0.0077
Inference-only standardization 40                          -0.0331  ...                    0.0395
                               80                          -0.0334  ...                    0.0388

[10 rows x 6 columns]

## Detection performance from saved CSVs

FNR, FPR, and AUC come directly from detection.csv and global_pmp_detection.csv. Component rows below show the median of the four saved candidate-model values for each diagnostic. Global PMP rows use each saved value once. AUC rank 1 is the highest among l2, linf, density, and mmd.


In [10]:
DIAGNOSTICS = ("l2", "linf", "density", "mmd")
detection_frames = []
for setting, result_dir in RESULT_DIRS.items():
    for dim in SUMMARY_DIMS:
        comparison_dir = result_dir / "comparison" / "n10" / f"indirect_s{dim}"
        for filename, scope in (
            ("detection.csv", "component"),
            ("global_pmp_detection.csv", "global_pmp"),
        ):
            frame = pd.read_csv(comparison_dir / filename)
            if (
                not frame["method"].eq("indirect").all()
                or not frame["scope"].eq(scope).all()
                or set(frame["diagnostic"]) != set(DIAGNOSTICS)
            ):
                raise ValueError(f"{comparison_dir / filename}: unexpected saved detection rows")
            detection_frames.append(
                frame.assign(setting=SETTING_LABELS[setting], summary_dim=dim)
            )

detection_data = pd.concat(detection_frames, ignore_index=True)
detection_comparison = (
    detection_data
    .groupby(["scope", "error_metric", "setting", "summary_dim", "diagnostic"], sort=False)
    [["FNR", "FPR", "AUC"]]
    .median()
    .unstack("diagnostic")
)
detection_comparison = detection_comparison.reindex(
    columns=pd.MultiIndex.from_product([("FNR", "FPR", "AUC"), DIAGNOSTICS])
)
table_index = pd.MultiIndex.from_product(
    [SETTING_LABELS.values(), SUMMARY_DIMS], names=["setting", "summary_dim"]
)


### Posterior MMD detection


In [11]:
display(
    detection_comparison.loc[("component", "posterior_mmd")]
    .reindex(table_index)
    .round(3)
)


FNR         ...     AUC       
                                               l2   linf  ... density    mmd
setting                        summary_dim                ...               
Default                        40           0.254  0.393  ...   0.988  0.879
                               80           0.108  0.383  ...   0.989  0.874
Different seed                 40           0.271  0.440  ...   0.980  0.877
                               80           0.103  0.466  ...   0.985  0.877
Fixed learning rate            40           0.195  0.335  ...   0.993  0.889
                               80           0.019  0.257  ...   0.994  0.880
No standardization             40           0.276  0.480  ...   0.990  0.880
                               80           0.047  0.380  ...   0.997  0.871
Summary-only standardization   40           0.257  0.398  ...   0.988  0.879
                               80           0.112  0.378  ...   0.984  0.875
Inference-only standardization 40           0.268  0.487  ...   0.989  0.878
                               80           0.041  0.410  ...   0.996  0.870

[12 rows x 12 columns]

### logML error detection


In [12]:
display(
    detection_comparison.loc[("component", "logml")]
    .reindex(table_index)
    .round(3)
)


FNR         ...     AUC       
                                               l2   linf  ... density    mmd
setting                        summary_dim                ...               
Default                        40           0.278  0.410  ...   0.963  0.852
                               80           0.121  0.390  ...   0.978  0.856
Different seed                 40           0.289  0.444  ...   0.947  0.831
                               80           0.127  0.467  ...   0.972  0.858
Fixed learning rate            40           0.210  0.333  ...   0.967  0.855
                               80           0.027  0.260  ...   0.971  0.854
No standardization             40           0.306  0.489  ...   0.949  0.827
                               80           0.068  0.390  ...   0.969  0.810
Summary-only standardization   40           0.280  0.414  ...   0.954  0.846
                               80           0.127  0.381  ...   0.976  0.855
Inference-only standardization 40           0.313  0.491  ...   0.931  0.820
                               80           0.064  0.425  ...   0.975  0.830

[12 rows x 12 columns]

### Candidate PMP error detection


In [13]:
display(
    detection_comparison.loc[("component", "pmp")]
    .reindex(table_index)
    .round(3)
)


FNR         ...     AUC       
                                               l2   linf  ... density    mmd
setting                        summary_dim                ...               
Default                        40           0.183  0.368  ...   0.555  0.634
                               80           0.106  0.362  ...   0.528  0.676
Different seed                 40           0.169  0.411  ...   0.564  0.663
                               80           0.179  0.440  ...   0.472  0.679
Fixed learning rate            40           0.229  0.344  ...   0.562  0.658
                               80           0.088  0.352  ...   0.623  0.661
No standardization             40           0.266  0.427  ...   0.564  0.668
                               80           0.115  0.396  ...   0.491  0.673
Summary-only standardization   40           0.176  0.374  ...   0.553  0.622
                               80           0.135  0.347  ...   0.509  0.646
Inference-only standardization 40           0.263  0.437  ...   0.570  0.662
                               80           0.110  0.422  ...   0.488  0.669

[12 rows x 12 columns]

### Dataset-level PMP error detection


In [14]:
display(
    detection_comparison.loc[("global_pmp", "pmp")]
    .reindex(table_index)
    .round(3)
)


FNR         ...     AUC       
                                             l2   linf  ... density    mmd
setting                        summary_dim              ...               
Default                        40           0.0  0.014  ...   0.719  0.658
                               80           0.0  0.000  ...   0.517  0.661
Different seed                 40           0.0  0.003  ...   0.615  0.699
                               80           0.0  0.017  ...   0.447  0.752
Fixed learning rate            40           0.0  0.004  ...   0.666  0.648
                               80           0.0  0.000  ...   0.545  0.629
No standardization             40           0.0  0.000  ...   0.614  0.655
                               80           0.0  0.000  ...   0.423  0.656
Summary-only standardization   40           0.0  0.000  ...   0.693  0.701
                               80           0.0  0.000  ...   0.601  0.719
Inference-only standardization 40           0.0  0.000  ...   0.635  0.686
                               80           0.0  0.000  ...   0.376  0.694

[12 rows x 12 columns]

### Density AUC rank across settings


In [15]:
density_ranks = detection_comparison["AUC"].rank(axis=1, ascending=False, method="min")
density_rank_table = density_ranks["density"].unstack(["setting", "summary_dim"])
density_rank_table = density_rank_table.reindex(
    columns=pd.MultiIndex.from_product(
        [SETTING_LABELS.values(), SUMMARY_DIMS], names=["setting", "summary_dim"]
    )
)
display(density_rank_table.astype(int))


setting                  Default     ... Inference-only standardization   
summary_dim                   40 80  ...                             40 80
scope      error_metric              ...                                  
component  posterior_mmd       1  1  ...                              1  1
           logml               1  1  ...                              1  1
           pmp                 4  4  ...                              3  4
global_pmp pmp                 1  3  ...                              2  4

[4 rows x 12 columns]